In [1]:
import xarray as xr 
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline

In [2]:
ds = xr.open_dataset('../data/alldata_tropicalAtl.nc')
zos = ds['zos']
ddtzos = (zos.roll(time=-1) - zos)/(24*3600)
ds['wtop'] = ddtzos
ds.wtop.attrs = {'units':'m/s',
                 'long_name': 'estimate of vertical velocity at top calculated from differences of SSH'}

In [3]:
def getUpwellIngVel(u, v , wtop, dx, dy, mld):
    taxis = 0
    yaxis = 1
    xaxis = 2
    
    x_p = 0.5 *(np.roll(u, -1, axis=xaxis) + u) 
    x_m = 0.5 *(np.roll(u, 1, axis=xaxis) + u)
    
    y_p = 0.5 *(np.roll(v, -1, axis=yaxis) + v)
    y_m = 0.5 *(np.roll(v, 1, axis=yaxis) + v)
    grad_x = (x_p -x_m)/(dx)
    grad_y = (y_p -y_m)/(dy)
    horiz_div = grad_x + grad_y
    w = wtop + mld * horiz_div
    return w

In [5]:
uo = ds['uo']
vo = ds['vo']

earthRad = 6378137 #6.371e6
lat = ds['latitude'].to_numpy()
lon = ds['longitude'].to_numpy()
depth = ds['depth'].to_numpy()
    
xlen = len(lon)
ylen = len(lat)
zlen = len(depth)
    
dx = np.zeros((ylen, xlen), dtype=float)
dy = np.zeros((ylen, xlen), dtype=float)

depth3d = np.zeros((zlen, ylen, xlen), dtype=float)
for i in range(zlen):
    depth3d[i,:,:] = depth[i]

depthXar = xr.DataArray(depth3d, dims=['depth', 'latitude', 'longitude'],   
                        attrs= {'units':  'm', 'long_name': '3d array of depth'}, 
                        coords = {'depth': ds['depth'],
                                  'latitude':ds['latitude'],
                                  'longitude': ds['longitude']})

#change from deg to radians
lat = np.deg2rad(lat)
lon = np.deg2rad(lon)
    
dlon = abs(lon[1] - lon[0])
dlat = abs(lat[1] - lat[0])

for i in range(ylen):
    R = earthRad * np.cos(lat[i])
    dx[i,:] = R*dlon
    dy[i,:] = earthRad * dlat

ds['depth3d'] = depthXar

uo = ds['uo']
vo = ds['vo']
wtop = ds['wtop']
mld = ds['mlotst']

#mask where depth is greater than mixed layer depth
mldMask = ds['depth'] > mld
uo = xr.where(mldMask, np.nan, uo)
vo = xr.where(mldMask, np.nan, vo)
depthXar = xr.where(mldMask, np.nan, depthXar)

#mask where nan and unphyiscal values for u and v
mask = np.isnan(uo.to_numpy())
mask = np.logical_or(mask, np.isnan(vo.to_numpy() ))
mask = np.logical_or(mask, abs(uo.to_numpy())> 100)
mask = np.logical_or(mask, abs(vo.to_numpy())> 100)
uo = xr.where(mask, np.nan, uo)
vo = xr.where(mask, np.nan, vo)

#depth averaging
uoDepth = uo*depthXar
voDepth = vo*depthXar
mldArr = depthXar.sum(dim='depth')

avUo = uoDepth.sum(dim='depth')/depthXar.sum(dim='depth')
avVo = voDepth.sum(dim='depth')/depthXar.sum(dim='depth')

w = getUpwellIngVel(avUo.to_numpy(), 
                    avVo.to_numpy(), 
                    wtop.to_numpy(), 
                    dx, dy,  
                    mldArr.to_numpy())

dimsList = ('time','latitude', 'longitude')


wbase = xr.DataArray(w, 
                     dims=dimsList,   
                     attrs= {'units':  'm/s', 
                             'long_name':'vertical velcity at base of mixed layer' })

ds['wbot'] = wbase

In [6]:
ds.to_netcdf('../data/TropAtl_MLD_topAndBotVertVel.nc')